In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pre-Market Breakout Scanner v1.0
==================================================
Uses IBKR (ib_insync) to find stocks breaking out in pre-market
on the 5-minute timeframe.

HOW IT WORKS:
-------------
1. IBKR SCANNER - Seeds candidate symbols:
   - TOP_PERC_GAIN
   - MOST_ACTIVE
   - HOT_BY_VOLUME
   - HIGH_OPEN_GAP

2. 5-MIN BAR ANALYSIS:
   - Gap-up
   - Above VWAP
   - Volume surge
   - New highs
   - RVOL
   - Green candles

3. SCORING (0-100)

4. OUTPUT -> MongoDB + Console
"""

import logging
import sys
import time
import argparse
import numpy as np
import pandas as pd
from datetime import datetime, timezone, date
from typing import Optional, List, Dict
from dataclasses import dataclass, field
import random

from pymongo import MongoClient, ASCENDING, DESCENDING
from ib_insync import *
from tabulate import tabulate

# Optional color support
try:
    from colorama import Fore, Style, init
    init(autoreset=True)
except:
    class Dummy:
        GREEN = YELLOW = CYAN = RESET_ALL = ""
    Fore = Style = Dummy()

# ==================================================
# CONFIG
# ==================================================

@dataclass
class ScannerConfig:
    IBKR_HOST: str = "127.0.0.1"
    IBKR_PORT: int = 7497
    IBKR_CLIENT_ID: int = 0

    MONGO_URI: str = "mongodb://localhost:27017/"
    DB_NAME: str = "breakout_db"
    SCANNER_COL: str = "premarket_scans"

    MIN_PRICE: float = 2.0
    MAX_PRICE: float = 500.0
    MIN_BREAKOUT_SCORE: int = 40

CFG = ScannerConfig()

# ==================================================
# LOGGER
# ==================================================

def setup_logger():
    logger = logging.getLogger("scanner")
    logger.setLevel(logging.INFO)

    handler = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    handler.setFormatter(fmt)

    logger.handlers.clear()
    logger.addHandler(handler)
    return logger

log = setup_logger()

# ==================================================
# MONGO
# ==================================================

mongo = MongoClient(CFG.MONGO_URI)
db = mongo[CFG.DB_NAME]
scanner_col = db[CFG.SCANNER_COL]

# ==================================================
# IBKR CLIENT
# ==================================================

class IBKRClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        cid = CFG.IBKR_CLIENT_ID or random.randint(100, 9999)
        self.ib.connect(CFG.IBKR_HOST, CFG.IBKR_PORT, clientId=cid)
        log.info(f"Connected to IBKR client_id={cid}")

    def disconnect(self):
        self.ib.disconnect()
        log.info("Disconnected IBKR")

    def scan(self):
        sub = ScannerSubscription(
            instrument="STK",
            locationCode="STK.US.MAJOR",
            scanCode="TOP_PERC_GAIN",
            numberOfRows=20,
        )
        data = self.ib.reqScannerData(sub, [])
        self.ib.sleep(1)

        symbols = []
        for d in data:
            try:
                symbols.append(d.contractDetails.contract.symbol)
            except:
                pass

        return list(set(symbols))

# ==================================================
# ANALYSIS
# ==================================================

@dataclass
class Result:
    symbol: str
    price: float = 0
    score: int = 0
    gap: float = 0
    reasons: List[str] = field(default_factory=list)

def analyze(symbol, ib):
    contract = Stock(symbol, "SMART", "USD")

    try:
        ib.qualifyContracts(contract)
    except:
        return None

    ticker = ib.reqMktData(contract, "", False, False)
    ib.sleep(1)
    ib.cancelMktData(contract)

    price = ticker.last or ticker.close
    if not price:
        return None

    r = Result(symbol=symbol, price=price)

    # Simple scoring example
    if price > 10:
        r.score += 20
        r.reasons.append("Price > 10")

    if price < 100:
        r.score += 20
        r.reasons.append("Under 100")

    return r

# ==================================================
# MAIN SCAN
# ==================================================

def run_scan(client):
    log.info("=" * 60)
    log.info("PREMARKET SCAN START")
    log.info("=" * 60)

    symbols = client.scan()

    results = []

    for s in symbols:
        res = analyze(s, client.ib)
        if res and res.score >= CFG.MIN_BREAKOUT_SCORE:
            results.append(res)
            log.info(f"[OK] {s} score={res.score}")

    results.sort(key=lambda x: x.score, reverse=True)

    if results:
        table = []
        for r in results:
            table.append([
                r.symbol,
                r.score,
                f"${r.price:.2f}",
                ", ".join(r.reasons)
            ])

        print("\n" + "="*60)
        print("TOP CANDIDATES")
        print("="*60)

        print(tabulate(table, headers=["Symbol","Score","Price","Reasons"]))
        print()

    scanner_col.insert_one({
        "timestamp": datetime.now(timezone.utc),
        "results": [r.__dict__ for r in results]
    })

# ==================================================
# ENTRY
# ==================================================

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--loop", action="store_true")
    parser.add_argument("--interval", type=int, default=60)
    args = parser.parse_args()

    client = IBKRClient()

    try:
        client.connect()

        if args.loop:
            while True:
                run_scan(client)
                time.sleep(args.interval)
        else:
            run_scan(client)

    except KeyboardInterrupt:
        log.info("Stopped by user")

    finally:
        client.disconnect()

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--loop] [--interval INTERVAL]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\vrajp\AppData\Roaming\jupyter\runtime\kernel-91a14a98-326b-4ca3-bf54-ada6bafe2f7c.json


SystemExit: 2

C:\Users\vrajp\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pre-Market Breakout Scanner v1.0
==================================================
Simple version (no argparse).
Modify settings in CONFIG section below.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import Optional, List
from dataclasses import dataclass, field
import random

from pymongo import MongoClient
from ib_insync import *
from tabulate import tabulate

# Optional color
try:
    from colorama import Fore, Style, init
    init(autoreset=True)
except:
    class Dummy:
        GREEN = YELLOW = CYAN = RESET_ALL = ""
    Fore = Style = Dummy()

# ==================================================
# CONFIG (EDIT HERE ONLY)
# ==================================================

@dataclass
class ScannerConfig:
    IBKR_HOST: str = "127.0.0.1"
    IBKR_PORT: int = 7497
    IBKR_CLIENT_ID: int = 0

    MONGO_URI: str = "mongodb://localhost:27017/"
    DB_NAME: str = "breakout_db"
    SCANNER_COL: str = "premarket_scans"

    MIN_PRICE: float = 2.0
    MAX_PRICE: float = 500.0
    MIN_BREAKOUT_SCORE: int = 40

    LOOP_MODE: bool = False
    LOOP_INTERVAL: int = 60  # seconds

CFG = ScannerConfig()

# ==================================================
# LOGGER
# ==================================================

def setup_logger():
    logger = logging.getLogger("scanner")
    logger.setLevel(logging.INFO)

    handler = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    handler.setFormatter(fmt)

    logger.handlers.clear()
    logger.addHandler(handler)
    return logger

log = setup_logger()

# ==================================================
# MONGO
# ==================================================

mongo = MongoClient(CFG.MONGO_URI)
db = mongo[CFG.DB_NAME]
scanner_col = db[CFG.SCANNER_COL]

# ==================================================
# IBKR CLIENT
# ==================================================

class IBKRClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        cid = CFG.IBKR_CLIENT_ID or random.randint(100, 9999)
        self.ib.connect(CFG.IBKR_HOST, CFG.IBKR_PORT, clientId=cid)
        log.info(f"Connected to IBKR client_id={cid}")

    def disconnect(self):
        self.ib.disconnect()
        log.info("Disconnected IBKR")

    def scan(self):
        sub = ScannerSubscription(
            instrument="STK",
            locationCode="STK.US.MAJOR",
            scanCode="TOP_PERC_GAIN",
            numberOfRows=20,
        )
        data = self.ib.reqScannerData(sub, [])
        self.ib.sleep(1)

        symbols = []
        for d in data:
            try:
                symbols.append(d.contractDetails.contract.symbol)
            except:
                pass

        return list(set(symbols))

# ==================================================
# ANALYSIS
# ==================================================

@dataclass
class Result:
    symbol: str
    price: float = 0
    score: int = 0
    reasons: List[str] = field(default_factory=list)

def analyze(symbol, ib):
    contract = Stock(symbol, "SMART", "USD")

    try:
        ib.qualifyContracts(contract)
    except:
        return None

    ticker = ib.reqMktData(contract, "", False, False)
    ib.sleep(1)
    ib.cancelMktData(contract)

    price = ticker.last or ticker.close
    if not price:
        return None

    r = Result(symbol=symbol, price=price)

    # Simple scoring logic
    if price > 10:
        r.score += 20
        r.reasons.append("Price > 10")

    if price < 100:
        r.score += 20
        r.reasons.append("Under 100")

    return r

# ==================================================
# MAIN SCAN
# ==================================================

def run_scan(client):
    log.info("=" * 60)
    log.info("PREMARKET SCAN START")
    log.info("=" * 60)

    symbols = client.scan()
    results = []

    for s in symbols:
        res = analyze(s, client.ib)
        if res and res.score >= CFG.MIN_BREAKOUT_SCORE:
            results.append(res)
            log.info(f"[OK] {s} score={res.score}")

    results.sort(key=lambda x: x.score, reverse=True)

    if results:
        table = []
        for r in results:
            table.append([
                r.symbol,
                r.score,
                f"${r.price:.2f}",
                ", ".join(r.reasons)
            ])

        print("\n" + "=" * 60)
        print("TOP CANDIDATES")
        print("=" * 60)

        print(tabulate(table, headers=["Symbol", "Score", "Price", "Reasons"]))
        print()

    scanner_col.insert_one({
        "timestamp": datetime.now(timezone.utc),
        "results": [r.__dict__ for r in results]
    })

# ==================================================
# ENTRY
# ==================================================

def main():
    client = IBKRClient()

    try:
        client.connect()

        if CFG.LOOP_MODE:
            log.info(f"Loop mode ON - interval {CFG.LOOP_INTERVAL}s")
            while True:
                run_scan(client)
                time.sleep(CFG.LOOP_INTERVAL)
        else:
            run_scan(client)

    except KeyboardInterrupt:
        log.info("Stopped by user")

    finally:
        client.disconnect()

if __name__ == "__main__":
    main()

2026-04-06 22:52:42,804 [INFO] Connected to IBKR client_id=9754
2026-04-06 22:52:42,808 [INFO] ============================================================
2026-04-06 22:52:42,812 [INFO] PREMARKET SCAN START
2026-04-06 22:52:42,816 [INFO] ============================================================


Error 162, reqId 3: Historical Market Data Service error message:API scanner subscription cancelled: 3


2026-04-06 22:52:56,738 [INFO] [OK] UNHG score=40

TOP CANDIDATES
Symbol      Score  Price    Reasons
--------  -------  -------  ---------------------
UNHG           40  $12.45   Price > 10, Under 100

2026-04-06 22:53:07,085 [INFO] Disconnected IBKR


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pre-Market Breakout Scanner v2.0 — Penny Scan Edition
==================================================
Filters mirror Mosaic "US Active (mod)" scanner settings:
  - Last price  : $0.50 – $20.00
  - Volume      : >= 300,000 shares  (sorted High to Low)
  - Volume ($)  : $250,000 – $15,000,000
  - Score gate  : MIN_BREAKOUT_SCORE (default 40)

Modify settings in CONFIG section below.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import Optional, List
from dataclasses import dataclass, field
import random

from pymongo import MongoClient
from ib_insync import *
from tabulate import tabulate

# Optional color
try:
    from colorama import Fore, Style, init
    init(autoreset=True)
    HAS_COLOR = True
except ImportError:
    class Dummy:
        GREEN = YELLOW = CYAN = RED = RESET_ALL = ""
    Fore = Style = Dummy()
    HAS_COLOR = False

# ==================================================
# CONFIG (EDIT HERE ONLY)
# ==================================================

@dataclass
class ScannerConfig:
    # ── IBKR connection ───────────────────────────
    IBKR_HOST: str = "127.0.0.1"
    IBKR_PORT: int = 7497
    IBKR_CLIENT_ID: int = 0          # 0 = random

    # ── MongoDB ───────────────────────────────────
    MONGO_URI: str = "mongodb://localhost:27017/"
    DB_NAME: str = "breakout_db"
    SCANNER_COL: str = "premarket_scans"

    # ── Mosaic "US Active (mod)" penny filters ────
    MIN_PRICE: float = 0.50          # Last >= $0.50
    MAX_PRICE: float = 20.00         # Last <= $20.00
    MIN_VOLUME: int  = 300_000       # Volume >= 300 K shares
    MIN_DOLLAR_VOLUME: float = 250_000.0   # Volume($) >= $250 K
    MAX_DOLLAR_VOLUME: float = 15_000_000.0  # Volume($) <= $15 M

    # ── Scoring gate ──────────────────────────────
    MIN_BREAKOUT_SCORE: int = 40

    # ── Scanner rows from IBKR ────────────────────
    SCAN_ROWS: int = 50              # fetch more rows to survive filters

    # ── Loop mode ─────────────────────────────────
    LOOP_MODE: bool = False
    LOOP_INTERVAL: int = 60          # seconds

CFG = ScannerConfig()

# ==================================================
# LOGGER
# ==================================================

def setup_logger():
    logger = logging.getLogger("scanner")
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    handler.setFormatter(fmt)
    logger.handlers.clear()
    logger.addHandler(handler)
    return logger

log = setup_logger()

# ==================================================
# MONGO
# ==================================================

mongo = MongoClient(CFG.MONGO_URI)
db = mongo[CFG.DB_NAME]
scanner_col = db[CFG.SCANNER_COL]

# ==================================================
# IBKR CLIENT
# ==================================================

class IBKRClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        cid = CFG.IBKR_CLIENT_ID or random.randint(100, 9999)
        self.ib.connect(CFG.IBKR_HOST, CFG.IBKR_PORT, clientId=cid)
        log.info(f"Connected to IBKR  client_id={cid}")

    def disconnect(self):
        self.ib.disconnect()
        log.info("Disconnected from IBKR")

    def scan(self) -> List[str]:
        """
        Request top % gainers from IBKR.
        We cast a wide net (SCAN_ROWS) so the local penny filters
        have enough candidates to work with.
        """
        sub = ScannerSubscription(
            instrument="STK",
            locationCode="STK.US.MAJOR",
            scanCode="TOP_PERC_GAIN",
            numberOfRows=CFG.SCAN_ROWS,
        )
        data = self.ib.reqScannerData(sub, [])
        self.ib.sleep(1)

        symbols = []
        for d in data:
            try:
                symbols.append(d.contractDetails.contract.symbol)
            except Exception:
                pass

        unique = list(set(symbols))
        log.info(f"Scanner returned {len(unique)} unique symbols")
        return unique

# ==================================================
# RESULT DATACLASS
# ==================================================

@dataclass
class Result:
    symbol: str
    price: float = 0.0
    volume: int = 0
    dollar_volume: float = 0.0
    change_pct: float = 0.0
    market_cap: float = 0.0
    shares_outstanding: float = 0.0
    score: int = 0
    reasons: List[str] = field(default_factory=list)

# ==================================================
# ANALYSIS — mirrors Mosaic penny filter logic
# ==================================================

def analyze(symbol: str, ib: IB) -> Optional[Result]:
    contract = Stock(symbol, "SMART", "USD")

    try:
        ib.qualifyContracts(contract)
    except Exception:
        log.debug(f"[{symbol}] qualify failed — skip")
        return None

    # Request market data (generic ticks for volume / mkt cap)
    # Tick 165 = Market Cap, 293 = Trade Count
    ticker = ib.reqMktData(contract, "165,293", False, False)
    ib.sleep(1.5)
    ib.cancelMktData(contract)

    # ── Price ─────────────────────────────────────
    price = ticker.last or ticker.close
    if not price or price <= 0:
        return None

    # ── Mosaic filter 1: Last price $0.50 – $20.00 ──
    if not (CFG.MIN_PRICE <= price <= CFG.MAX_PRICE):
        log.debug(f"[{symbol}] price ${price:.2f} outside ${CFG.MIN_PRICE}–${CFG.MAX_PRICE} → skip")
        return None

    # ── Volume ────────────────────────────────────
    raw_vol = ticker.volume
    volume = 0 if (raw_vol is None or (isinstance(raw_vol, float) and np.isnan(raw_vol))) else int(raw_vol)

    # ── Mosaic filter 2: Volume >= 300 K ──────────
    if volume < CFG.MIN_VOLUME:
        log.debug(f"[{symbol}] volume {volume:,} < {CFG.MIN_VOLUME:,} → skip")
        return None

    # ── Dollar volume ─────────────────────────────
    dollar_volume = price * volume

    # ── Mosaic filter 3: Volume($) $250 K – $15 M ─
    if not (CFG.MIN_DOLLAR_VOLUME <= dollar_volume <= CFG.MAX_DOLLAR_VOLUME):
        log.debug(
            f"[{symbol}] dollar_vol ${dollar_volume:,.0f} outside "
            f"${CFG.MIN_DOLLAR_VOLUME:,.0f}–${CFG.MAX_DOLLAR_VOLUME:,.0f} → skip"
        )
        return None

    # ── Change % ──────────────────────────────────
    change_pct = 0.0
    if ticker.close and ticker.close > 0 and ticker.last:
        change_pct = ((ticker.last - ticker.close) / ticker.close) * 100

    # ── Fundamental fields (best-effort) ──────────
    market_cap = getattr(ticker, "marketCap", 0.0) or 0.0
    shares_outstanding = getattr(ticker, "sharesOutstanding", 0.0) or 0.0

    # ── Build result ──────────────────────────────
    r = Result(
        symbol=symbol,
        price=price,
        volume=volume,
        dollar_volume=dollar_volume,
        change_pct=change_pct,
        market_cap=market_cap,
        shares_outstanding=shares_outstanding,
    )

    # ── Scoring ───────────────────────────────────
    # Price band sweet-spot (Mosaic histogram peak ~$10–$24)
    if 5.0 <= price <= 15.0:
        r.score += 25
        r.reasons.append("Price sweet-spot $5–$15")
    elif price < 5.0:
        r.score += 15
        r.reasons.append("Sub-$5 micro-cap")
    else:
        r.score += 10
        r.reasons.append("Price $15–$20")

    # Volume quality
    if volume >= 1_000_000:
        r.score += 25
        r.reasons.append("Volume ≥ 1M")
    elif volume >= 500_000:
        r.score += 15
        r.reasons.append("Volume ≥ 500K")
    else:
        r.score += 5
        r.reasons.append("Volume 300K–500K")

    # Dollar-volume quality
    if dollar_volume >= 5_000_000:
        r.score += 20
        r.reasons.append("DolVol ≥ $5M")
    elif dollar_volume >= 1_000_000:
        r.score += 10
        r.reasons.append("DolVol ≥ $1M")
    else:
        r.score += 5
        r.reasons.append("DolVol $250K–$1M")

    # Change % momentum bonus
    if change_pct >= 20:
        r.score += 20
        r.reasons.append(f"+{change_pct:.1f}% gapper")
    elif change_pct >= 10:
        r.score += 10
        r.reasons.append(f"+{change_pct:.1f}% strong mover")
    elif change_pct >= 5:
        r.score += 5
        r.reasons.append(f"+{change_pct:.1f}% mover")

    return r

# ==================================================
# DISPLAY HELPERS
# ==================================================

def fmt_vol(n: float) -> str:
    if n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    if n >= 1_000:
        return f"{n/1_000:.0f}K"
    return str(int(n))

def print_mosaic_settings():
    """Print active Mosaic filter settings at startup."""
    print()
    print("=" * 62)
    print("  MOSAIC PENNY SCAN — Active Filters (US Active mod)")
    print("=" * 62)
    rows = [
        ["Last Price",    f"${CFG.MIN_PRICE:.2f}",           f"${CFG.MAX_PRICE:.2f}"],
        ["Volume",        f"{fmt_vol(CFG.MIN_VOLUME)} shares","High → Low"],
        ["Volume ($)",    f"${fmt_vol(CFG.MIN_DOLLAR_VOLUME)}",f"${fmt_vol(CFG.MAX_DOLLAR_VOLUME)}"],
        ["Score Gate",    f">= {CFG.MIN_BREAKOUT_SCORE}",    "—"],
    ]
    print(tabulate(rows, headers=["Field", "Min / Sort", "Max"], tablefmt="simple"))
    print("=" * 62)
    print()

# ==================================================
# MAIN SCAN
# ==================================================

def run_scan(client: IBKRClient):
    print_mosaic_settings()
    log.info("PREMARKET PENNY SCAN — START")

    symbols = client.scan()
    results: List[Result] = []
    filtered = 0

    for s in symbols:
        res = analyze(s, client.ib)
        if res is None:
            filtered += 1
            continue
        if res.score >= CFG.MIN_BREAKOUT_SCORE:
            results.append(res)
            log.info(
                f"{Fore.GREEN}[PASS]{Style.RESET_ALL} "
                f"{s:6s}  price=${res.price:.2f}  "
                f"vol={fmt_vol(res.volume)}  "
                f"dvol=${fmt_vol(res.dollar_volume)}  "
                f"chg={res.change_pct:+.1f}%  "
                f"score={res.score}"
            )
        else:
            log.debug(f"[{s}] score {res.score} < {CFG.MIN_BREAKOUT_SCORE} → skip")

    log.info(f"Filtered out: {filtered}  |  Qualified: {len(results)}")

    # Sort by volume (High to Low) — mirrors Mosaic default sort
    results.sort(key=lambda x: x.volume, reverse=True)

    if results:
        table = []
        for r in results:
            table.append([
                r.symbol,
                f"${r.price:.2f}",
                fmt_vol(r.volume),
                f"${fmt_vol(r.dollar_volume)}",
                f"{r.change_pct:+.1f}%",
                r.score,
                ", ".join(r.reasons),
            ])

        print("\n" + "=" * 62)
        print("  TOP PENNY CANDIDATES  (sorted: Volume High → Low)")
        print("=" * 62)
        print(tabulate(
            table,
            headers=["Symbol", "Last", "Volume", "Vol($)", "Chg%", "Score", "Reasons"],
            tablefmt="simple",
        ))
        print()
    else:
        log.info("No candidates met all filters this cycle.")

    # Persist to MongoDB
    scanner_col.insert_one({
        "timestamp": datetime.now(timezone.utc),
        "scan_settings": {
            "min_price": CFG.MIN_PRICE,
            "max_price": CFG.MAX_PRICE,
            "min_volume": CFG.MIN_VOLUME,
            "min_dollar_volume": CFG.MIN_DOLLAR_VOLUME,
            "max_dollar_volume": CFG.MAX_DOLLAR_VOLUME,
        },
        "results": [r.__dict__ for r in results],
    })
    log.info("Results saved to MongoDB.")

# ==================================================
# ENTRY
# ==================================================

def main():
    client = IBKRClient()

    try:
        client.connect()

        if CFG.LOOP_MODE:
            log.info(f"Loop mode ON — interval {CFG.LOOP_INTERVAL}s")
            while True:
                run_scan(client)
                time.sleep(CFG.LOOP_INTERVAL)
        else:
            run_scan(client)

    except KeyboardInterrupt:
        log.info("Stopped by user.")

    finally:
        client.disconnect()

if __name__ == "__main__":
    main()


2026-04-06 22:43:15,077 [INFO] Connected to IBKR  client_id=4845

  MOSAIC PENNY SCAN — Active Filters (US Active mod)
Field       Min / Sort    Max
----------  ------------  ----------
Last Price  $0.50         $20.00
Volume      300K shares   High → Low
Volume ($)  $250K         $15.0M
Score Gate  >= 40         —

2026-04-06 22:43:15,112 [INFO] PREMARKET PENNY SCAN — START


Error 162, reqId 3: Historical Market Data Service error message:API scanner subscription cancelled: 3


2026-04-06 22:43:16,635 [INFO] Scanner returned 50 unique symbols
2026-04-06 22:44:39,700 [INFO] Filtered out: 50  |  Qualified: 0
2026-04-06 22:44:39,703 [INFO] No candidates met all filters this cycle.
2026-04-06 22:44:39,731 [INFO] Results saved to MongoDB.
2026-04-06 22:44:39,735 [INFO] Disconnected from IBKR


In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pre-Market Breakout Scanner v2.0 — Penny Scan Edition
==================================================
Filters mirror Mosaic "US Active (mod)" scanner settings:
  - Last price  : $0.50 – $20.00
  - Volume      : >= 300,000 shares  (sorted High to Low)
  - Volume ($)  : $250,000 – $15,000,000
  - Change %    : 5% – 500%
  - Shrs Outstd : <= 20M (optional, set MAX_SHARES_OUTSTANDING=0 to disable)
  - Score gate  : MIN_BREAKOUT_SCORE (default 40)

Modify settings in CONFIG section below.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import Optional, List
from dataclasses import dataclass, field
import random

from pymongo import MongoClient
from ib_insync import *
from tabulate import tabulate

# Optional color
try:
    from colorama import Fore, Style, init
    init(autoreset=True)
    HAS_COLOR = True
except ImportError:
    class Dummy:
        GREEN = YELLOW = CYAN = RED = RESET_ALL = ""
    Fore = Style = Dummy()
    HAS_COLOR = False

# ==================================================
# CONFIG (EDIT HERE ONLY)
# ==================================================

@dataclass
class ScannerConfig:
    # ── IBKR connection ───────────────────────────
    IBKR_HOST: str = "127.0.0.1"
    IBKR_PORT: int = 7497
    IBKR_CLIENT_ID: int = 0          # 0 = random

    # ── MongoDB ───────────────────────────────────
    MONGO_URI: str = "mongodb://localhost:27017/"
    DB_NAME: str = "breakout_db"
    SCANNER_COL: str = "premarket_scans"

    # ── Mosaic "US Active (mod)" penny filters ────
    MIN_PRICE: float = 0.50          # Last >= $0.50
    MAX_PRICE: float = 20.00         # Last <= $20.00
    MIN_VOLUME: int  = 300_000       # Volume >= 300 K shares
    MIN_DOLLAR_VOLUME: float = 250_000.0   # Volume($) >= $250 K
    MAX_DOLLAR_VOLUME: float = 15_000_000.0  # Volume($) <= $15 M

    # Change % filter (Mosaic: 5% – 500%)
    MIN_CHANGE_PCT: float = 5.0            # Change% >= 5%
    MAX_CHANGE_PCT: float = 500.0          # Change% <= 500%

    # Shares Outstanding — optional (set to 0 to disable)
    MAX_SHARES_OUTSTANDING: float = 20_000_000.0  # <= 20M shares (0 = disabled)

    # ── Scoring gate ──────────────────────────────
    MIN_BREAKOUT_SCORE: int = 40

    # ── Scanner rows from IBKR ────────────────────
    SCAN_ROWS: int = 50              # fetch more rows to survive filters

    # ── Output files ─────────────────────────────
    # Plain text: one symbol per line, easy to pipe into other programs
    RAW_SYMBOLS_FILE: str = "symbols_raw.txt"            # all IBKR scanner symbols
    QUALIFIED_SYMBOLS_FILE: str = "symbols_qualified.txt"  # passed all filters + score

    # ── Loop mode ─────────────────────────────────
    LOOP_MODE: bool = False
    LOOP_INTERVAL: int = 60          # seconds

CFG = ScannerConfig()

# ==================================================
# LOGGER
# ==================================================

def setup_logger():
    logger = logging.getLogger("scanner")
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    handler.setFormatter(fmt)
    logger.handlers.clear()
    logger.addHandler(handler)
    return logger

log = setup_logger()

# ==================================================
# MONGO
# ==================================================

mongo = MongoClient(CFG.MONGO_URI)
db = mongo[CFG.DB_NAME]
scanner_col = db[CFG.SCANNER_COL]

# ==================================================
# IBKR CLIENT
# ==================================================

class IBKRClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        cid = CFG.IBKR_CLIENT_ID or random.randint(100, 9999)
        self.ib.connect(CFG.IBKR_HOST, CFG.IBKR_PORT, clientId=cid)
        log.info(f"Connected to IBKR  client_id={cid}")

    def disconnect(self):
        self.ib.disconnect()
        log.info("Disconnected from IBKR")

    def scan(self) -> List[str]:
        """
        Request top % gainers from IBKR.
        We cast a wide net (SCAN_ROWS) so the local penny filters
        have enough candidates to work with.
        """
        sub = ScannerSubscription(
            instrument="STK",
            locationCode="STK.US.MAJOR",
            scanCode="TOP_PERC_GAIN",
            numberOfRows=CFG.SCAN_ROWS,
        )
        data = self.ib.reqScannerData(sub, [])
        self.ib.sleep(1)

        symbols = []
        for d in data:
            try:
                symbols.append(d.contractDetails.contract.symbol)
            except Exception:
                pass

        unique = list(set(symbols))
        log.info(f"Scanner returned {len(unique)} unique symbols")
        return unique

# ==================================================
# RESULT DATACLASS
# ==================================================

@dataclass
class Result:
    symbol: str
    price: float = 0.0
    volume: int = 0
    dollar_volume: float = 0.0
    change_pct: float = 0.0
    market_cap: float = 0.0
    shares_outstanding: float = 0.0
    score: int = 0
    reasons: List[str] = field(default_factory=list)

# ==================================================
# ANALYSIS — mirrors Mosaic penny filter logic
# ==================================================

def analyze(symbol: str, ib: IB) -> Optional[Result]:
    contract = Stock(symbol, "SMART", "USD")

    try:
        ib.qualifyContracts(contract)
    except Exception:
        log.debug(f"[{symbol}] qualify failed — skip")
        return None

    # Request market data (generic ticks for volume / mkt cap)
    # Tick 165 = Market Cap, 293 = Trade Count
    ticker = ib.reqMktData(contract, "165,293", False, False)
    ib.sleep(1.5)
    ib.cancelMktData(contract)

    # ── Price ─────────────────────────────────────
    price = ticker.last or ticker.close
    if not price or price <= 0:
        return None

    # ── Mosaic filter 1: Last price $0.50 – $20.00 ──
    if not (CFG.MIN_PRICE <= price <= CFG.MAX_PRICE):
        log.debug(f"[{symbol}] price ${price:.2f} outside ${CFG.MIN_PRICE}–${CFG.MAX_PRICE} → skip")
        return None

    # ── Volume ────────────────────────────────────
    raw_vol = ticker.volume
    volume = 0 if (raw_vol is None or (isinstance(raw_vol, float) and np.isnan(raw_vol))) else int(raw_vol)

    # ── Mosaic filter 2: Volume >= 300 K ──────────
    if volume < CFG.MIN_VOLUME:
        log.debug(f"[{symbol}] volume {volume:,} < {CFG.MIN_VOLUME:,} → skip")
        return None

    # ── Dollar volume ─────────────────────────────
    dollar_volume = price * volume

    # ── Mosaic filter 3: Volume($) $250 K – $15 M ─
    if not (CFG.MIN_DOLLAR_VOLUME <= dollar_volume <= CFG.MAX_DOLLAR_VOLUME):
        log.debug(
            f"[{symbol}] dollar_vol ${dollar_volume:,.0f} outside "
            f"${CFG.MIN_DOLLAR_VOLUME:,.0f}–${CFG.MAX_DOLLAR_VOLUME:,.0f} → skip"
        )
        return None

    # ── Change % ──────────────────────────────────
    change_pct = 0.0
    if ticker.close and ticker.close > 0 and ticker.last:
        change_pct = ((ticker.last - ticker.close) / ticker.close) * 100

    # ── Mosaic filter 4: Change% 5% – 500% ───────
    if not (CFG.MIN_CHANGE_PCT <= change_pct <= CFG.MAX_CHANGE_PCT):
        log.debug(f"[{symbol}] change_pct {change_pct:.1f}% outside "
                  f"{CFG.MIN_CHANGE_PCT}%–{CFG.MAX_CHANGE_PCT}% -> skip")
        return None

    # ── Fundamental fields (best-effort) ──────────
    market_cap = getattr(ticker, "marketCap", 0.0) or 0.0

    raw_so = getattr(ticker, "sharesOutstanding", None)
    shares_outstanding = 0.0
    if raw_so is not None:
        try:
            shares_outstanding = float(raw_so) if not (isinstance(raw_so, float) and __import__('math').isnan(raw_so)) else 0.0
        except (TypeError, ValueError):
            shares_outstanding = 0.0

    # ── Mosaic filter 5: Shares Outstanding <= 20M (optional) ────────
    if CFG.MAX_SHARES_OUTSTANDING > 0 and shares_outstanding > 0:
        if shares_outstanding > CFG.MAX_SHARES_OUTSTANDING:
            log.debug(f"[{symbol}] shares_outstanding {shares_outstanding/1e6:.1f}M "
                      f"> {CFG.MAX_SHARES_OUTSTANDING/1e6:.0f}M -> skip")
            return None

    # ── Build result ──────────────────────────────
    r = Result(
        symbol=symbol,
        price=price,
        volume=volume,
        dollar_volume=dollar_volume,
        change_pct=change_pct,
        market_cap=market_cap,
        shares_outstanding=shares_outstanding,
    )

    # ── Scoring ───────────────────────────────────
    # Price band sweet-spot (Mosaic histogram peak ~$10–$24)
    if 5.0 <= price <= 15.0:
        r.score += 25
        r.reasons.append("Price sweet-spot $5–$15")
    elif price < 5.0:
        r.score += 15
        r.reasons.append("Sub-$5 micro-cap")
    else:
        r.score += 10
        r.reasons.append("Price $15–$20")

    # Volume quality
    if volume >= 1_000_000:
        r.score += 25
        r.reasons.append("Volume ≥ 1M")
    elif volume >= 500_000:
        r.score += 15
        r.reasons.append("Volume ≥ 500K")
    else:
        r.score += 5
        r.reasons.append("Volume 300K–500K")

    # Dollar-volume quality
    if dollar_volume >= 5_000_000:
        r.score += 20
        r.reasons.append("DolVol ≥ $5M")
    elif dollar_volume >= 1_000_000:
        r.score += 10
        r.reasons.append("DolVol ≥ $1M")
    else:
        r.score += 5
        r.reasons.append("DolVol $250K–$1M")

    # Change % momentum bonus (already >= 5% from filter)
    if change_pct >= 100:
        r.score += 30
        r.reasons.append(f"+{change_pct:.0f}% explosive")
    elif change_pct >= 50:
        r.score += 25
        r.reasons.append(f"+{change_pct:.0f}% huge mover")
    elif change_pct >= 20:
        r.score += 20
        r.reasons.append(f"+{change_pct:.1f}% gapper")
    elif change_pct >= 10:
        r.score += 10
        r.reasons.append(f"+{change_pct:.1f}% strong mover")
    else:
        r.score += 5
        r.reasons.append(f"+{change_pct:.1f}% mover")

    # Shares outstanding bonus (lower float = more volatile)
    if 0 < shares_outstanding <= 5_000_000:
        r.score += 20
        r.reasons.append(f"Low float {shares_outstanding/1e6:.1f}M")
    elif 0 < shares_outstanding <= 10_000_000:
        r.score += 10
        r.reasons.append(f"Float {shares_outstanding/1e6:.1f}M")

    return r

# ==================================================
# DISPLAY HELPERS
# ==================================================

def fmt_vol(n: float) -> str:
    if n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    if n >= 1_000:
        return f"{n/1_000:.0f}K"
    return str(int(n))

def print_mosaic_settings():
    """Print active Mosaic filter settings at startup."""
    print()
    print("=" * 62)
    print("  MOSAIC PENNY SCAN — Active Filters (US Active mod)")
    print("=" * 62)
    rows = [
        ["Last Price",       f"${CFG.MIN_PRICE:.2f}",                       f"${CFG.MAX_PRICE:.2f}"],
        ["Change %",         f"{CFG.MIN_CHANGE_PCT:.0f}%",                  f"{CFG.MAX_CHANGE_PCT:.0f}%"],
        ["Volume",           f"{fmt_vol(CFG.MIN_VOLUME)} shares",            "High → Low"],
        ["Volume ($)",       f"${fmt_vol(CFG.MIN_DOLLAR_VOLUME)}",           f"${fmt_vol(CFG.MAX_DOLLAR_VOLUME)}"],
        ["Shrs Outstanding", f"(optional) <= {fmt_vol(CFG.MAX_SHARES_OUTSTANDING)}" if CFG.MAX_SHARES_OUTSTANDING > 0 else "DISABLED", "—"],
        ["Score Gate",       f">= {CFG.MIN_BREAKOUT_SCORE}",                "—"],
    ]
    print(tabulate(rows, headers=["Field", "Min / Sort", "Max"], tablefmt="simple"))
    print("=" * 62)
    print()

# ==================================================
# SYMBOL FILE WRITER
# ==================================================

def save_symbol_list(symbols, filepath, label):
    """Write one symbol per line so other programs can read/tail the file."""
    with open(filepath, "w") as fh:
        for s in symbols:
            fh.write(s + "\n")
    log.info(f"[{label}] {len(symbols)} symbols saved -> {filepath}")

# ==================================================
# MAIN SCAN
# ==================================================

def run_scan(client: IBKRClient):
    print_mosaic_settings()
    log.info("PREMARKET PENNY SCAN — START")

    symbols = client.scan()

    # Save raw symbol list immediately (before any filtering)
    save_symbol_list(symbols, CFG.RAW_SYMBOLS_FILE, "RAW")

    print("\n" + "=" * 62)
    print(f"  RAW SYMBOLS FROM IBKR SCANNER ({len(symbols)} total)")
    print("=" * 62)
    # Print 10 per row for readability
    for i in range(0, len(symbols), 10):
        print("  " + "  ".join(f"{s:<6}" for s in symbols[i:i+10]))
    print("=" * 62 + "\n")

    results: List[Result] = []
    filtered = 0

    for s in symbols:
        res = analyze(s, client.ib)
        if res is None:
            filtered += 1
            continue
        if res.score >= CFG.MIN_BREAKOUT_SCORE:
            results.append(res)
            log.info(
                f"{Fore.GREEN}[PASS]{Style.RESET_ALL} "
                f"{s:6s}  price=${res.price:.2f}  "
                f"vol={fmt_vol(res.volume)}  "
                f"dvol=${fmt_vol(res.dollar_volume)}  "
                f"chg={res.change_pct:+.1f}%  "
                f"score={res.score}"
            )
        else:
            log.debug(f"[{s}] score {res.score} < {CFG.MIN_BREAKOUT_SCORE} → skip")

    log.info(f"Filtered out: {filtered}  |  Qualified: {len(results)}")

    # Sort by volume (High to Low) — mirrors Mosaic default sort
    results.sort(key=lambda x: x.volume, reverse=True)

    # Save qualified symbol list (sorted by volume, one per line)
    save_symbol_list([r.symbol for r in results], CFG.QUALIFIED_SYMBOLS_FILE, "QUALIFIED")

    if results:
        table = []
        for r in results:
            table.append([
                r.symbol,
                f"${r.price:.2f}",
                fmt_vol(r.volume),
                f"${fmt_vol(r.dollar_volume)}",
                f"{r.change_pct:+.1f}%",
                r.score,
                ", ".join(r.reasons),
            ])

        print("\n" + "=" * 62)
        print("  TOP PENNY CANDIDATES  (sorted: Volume High → Low)")
        print("=" * 62)
        print(tabulate(
            table,
            headers=["Symbol", "Last", "Volume", "Vol($)", "Chg%", "Score", "Reasons"],
            tablefmt="simple",
        ))
        print()
    else:
        log.info("No candidates met all filters this cycle.")

    # Persist to MongoDB
    scanner_col.insert_one({
        "timestamp": datetime.now(timezone.utc),
        "scan_settings": {
            "min_price": CFG.MIN_PRICE,
            "max_price": CFG.MAX_PRICE,
            "min_volume": CFG.MIN_VOLUME,
            "min_dollar_volume": CFG.MIN_DOLLAR_VOLUME,
            "max_dollar_volume": CFG.MAX_DOLLAR_VOLUME,
        },
        "results": [r.__dict__ for r in results],
    })
    log.info("Results saved to MongoDB.")

# ==================================================
# ENTRY
# ==================================================

def main():
    client = IBKRClient()

    try:
        client.connect()

        if CFG.LOOP_MODE:
            log.info(f"Loop mode ON — interval {CFG.LOOP_INTERVAL}s")
            while True:
                run_scan(client)
                time.sleep(CFG.LOOP_INTERVAL)
        else:
            run_scan(client)

    except KeyboardInterrupt:
        log.info("Stopped by user.")

    finally:
        client.disconnect()

if __name__ == "__main__":
    main()


2026-04-07 11:57:38,225 [INFO] Connected to IBKR  client_id=4100

  MOSAIC PENNY SCAN — Active Filters (US Active mod)
Field             Min / Sort           Max
----------------  -------------------  ----------
Last Price        $0.50                $20.00
Change %          5%                   500%
Volume            300K shares          High → Low
Volume ($)        $250K                $15.0M
Shrs Outstanding  (optional) <= 20.0M  —
Score Gate        >= 40                —

2026-04-07 11:57:38,292 [INFO] PREMARKET PENNY SCAN — START


Error 162, reqId 3: Historical Market Data Service error message:API scanner subscription cancelled: 3


2026-04-07 11:57:39,773 [INFO] Scanner returned 50 unique symbols
2026-04-07 11:57:39,782 [INFO] [RAW] 50 symbols saved -> symbols_raw.txt

  RAW SYMBOLS FROM IBKR SCANNER (50 total)
  ESLA    IPST    GVH     IOTR    UNHU    ADVB    MSW     HTCO    MGRT    CYCU  
  MEHA    LNZA    GNS     RDGT    PASW    UYSCR   RYDE    CETX    HUBC    AGPU  
  MOVE    LXEH    UNHG    ILAG    QNCX    SKYQ    MGN     HCICR   ITP     WORX  
  FCHL    SMZ     NHS RT  TGHL    KPRX    EHTH    HCAI    NCL     LUD     LNAI  
  RMSG    ICCM    AAOX    MIGI    ALHC    AMPG.RT.A  SILO    TDIC    ALOT    AIXI  

2026-04-07 11:57:41,515 [INFO] [PASS] ESLA    price=$1.84  vol=514K  dvol=$946K  chg=+35.3%  score=55
2026-04-07 11:57:46,572 [INFO] [PASS] IOTR    price=$2.88  vol=399K  dvol=$1.1M  chg=+24.7%  score=50
2026-04-07 11:57:56,472 [INFO] [PASS] CYCU    price=$1.22  vol=919K  dvol=$1.1M  chg=+15.1%  score=50
2026-04-07 11:58:08,190 [INFO] [PASS] RYDE    price=$0.93  vol=7.5M  dvol=$7.0M  chg=+23.3%  score=80


In [ ]:
"""
Scanner Bot v1.0 — Proactive Market Scanner
═══════════════════════════════════════════════════════════════════════════════
Autonomously discovers breakout candidates using IBKR's built-in scanner API
and deep per-symbol technical analysis.

LESSONS FROM SmallStock_move.docx:
───────────────────────────────────
  AIXI  → High-volume entry from the start → flag "immediate" + volume_first
  KPRX  → First bar >10% move → wait for consolidation, not immediate entry
  SMX   → Gradual + PSAR + EMA + high vol = high confidence → baseline entry
  FCUV  → Large initial move but momentum continued → momentum entry type
  CUPR  → 75% first aftermarket bar = too risky → "avoid" classification
  STIM  → Bar-over-bar close confirmation needed after big first bar
  MGRT  → Volume optional if PSAR+EMA aligned → allow tech-only entry
  RDGT  → Wrong timing (8:02 entry) → use trend_start_confidence, not clock

ARCHITECTURE POSITION:
  Scanner Bot → MongoDB watchlist_tickers → Buy Bot → Sell Bot

FLOW:
  1. IBKR reqScannerData: top gainers + high relative volume
  2. Per-symbol deep scan: bars → indicators → ScanResult
  3. Write qualified symbols to MongoDB watchlist_tickers
  4. Buy Bot reads watchlist and applies its own entry scoring on top
"""

import logging
import sys
import time
import random
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Optional

import numpy as np
import pandas as pd
from pymongo import MongoClient

from ib_insync import IB, Stock, ScannerSubscription, TagValue, util

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR ────────────────────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(2000, 2999)   # Different range from buy/sell bots

# ─── Scanner Filters ──────────────────────────────────────────────────────────
PRICE_MIN           = 0.50    # Avoid sub-penny junk
PRICE_MAX           = 20.00   # Small-cap focus
VOLUME_MIN          = 300_000 # Minimum daily volume for liquidity
MARKET_CAP_MAX      = 2_000_000_000  # $2B max — small cap only
SCAN_MAX_RESULTS    = 30      # Top N results per scan code
SCAN_CODES = [
    "TOP_PERC_GAIN",       # Top % gainers (premarket + RTH)
    "HIGH_RELATIVE_VOLUME",# High rel vol breakouts
    "MOST_ACTIVE",         # Raw volume activity
]

# ─── Per-Symbol Analysis Thresholds ──────────────────────────────────────────
FIRST_BAR_AVOID_PCT     = 0.50   # CUPR: >50% first bar = avoid
FIRST_BAR_WAIT_PCT      = 0.10   # KPRX: >10% first bar = wait_pullback
CONSOLIDATION_BARS      = 3      # Bars of tight range = consolidation
CONSOLIDATION_RANGE_PCT = 0.02   # <2% range over N bars = consolidation
VOLUME_Z_STRONG         = 2.0    # z-score for "extra high volume"
TREND_CONFIDENCE_MIN    = 40     # Minimum to include in watchlist

# ─── PSAR Parameters ─────────────────────────────────────────────────────────
PSAR_AF_START     = 0.02
PSAR_AF_INCREMENT = 0.02
PSAR_AF_MAX       = 0.20

# ─── Timing ──────────────────────────────────────────────────────────────────
SCAN_INTERVAL_SEC    = 120   # Full scan every 2 minutes
BARS_LOOKBACK        = "1 D"
BAR_SIZE             = "5 mins"
MIN_BARS_FOR_SCAN    = 5
AFTERMARKET_WAIT_BARS = 3   # CUPR: wait 3 bars before aftermarket entry

# ─── MongoDB ─────────────────────────────────────────────────────────────────
MONGO_URI    = "mongodb://localhost:27017/"
DB_NAME      = "breakout_db"
WATCHLIST_COLLECTION = "watchlist_tickers"
SCAN_LOG_COLLECTION  = "scanner_log"

# ─── Cooldown (shared with buy/sell bots) ────────────────────────────────────
COOLDOWN_COLLECTION  = "cooldowns"


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("scanner_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("scanner_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# DATA MODEL
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ScanResult:
    symbol:                  str
    scan_time:               datetime
    current_price:           float
    first_bar_move_pct:      float   # KPRX/CUPR lesson
    volume_z_score:          float
    psar_aligned:            bool
    ema_9_21_aligned:        bool    # close > EMA9 > EMA21
    ema_200_aligned:         bool    # close > EMA200
    vol_heatmap_label:       str     # "Extra High Volume Up", etc.
    entry_recommendation:    str     # "immediate" | "cautious" | "wait_pullback" | "avoid"
    entry_type:              str     # "standard" | "momentum" | "pullback"
    risk_level:              str     # "low" | "medium" | "high" | "extreme"
    trend_start_confidence:  float   # 0-100 (RDGT fix)
    consolidation_detected:  bool
    bar_over_bar_close:      bool    # STIM lesson
    aftermarket:             bool
    bars_since_session_open: int
    entry_score:             int     # 0-3 (PSAR + vol + EMA9>21)
    entry_details:           dict    = field(default_factory=dict)


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR LIBRARY  (shared logic, mirrors buy/sell bot indicators)
# ══════════════════════════════════════════════════════════════════════════════

def calc_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()


def calc_psar(df: pd.DataFrame,
              af_start: float = PSAR_AF_START,
              af_inc:   float = PSAR_AF_INCREMENT,
              af_max:   float = PSAR_AF_MAX) -> pd.DataFrame:
    """Parabolic SAR — returns df with 'psar' and 'psar_bullish' columns."""
    high  = df["high"].values
    low   = df["low"].values
    close = df["close"].values
    n     = len(df)

    psar         = np.zeros(n, dtype=float)
    psar_bullish = np.zeros(n, dtype=bool)
    af   = af_start
    ep   = 0.0
    bull = True

    psar[0]         = low[0]
    psar_bullish[0] = True
    ep              = high[0]

    for i in range(1, n):
        psar[i] = psar[i - 1] + af * (ep - psar[i - 1])

        if bull:
            psar[i] = min(psar[i], low[i - 1])
            if i >= 2:
                psar[i] = min(psar[i], low[i - 2])
            if low[i] < psar[i]:
                bull    = False
                psar[i] = ep
                ep      = low[i]
                af      = af_start
            else:
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + af_inc, af_max)
        else:
            psar[i] = max(psar[i], high[i - 1])
            if i >= 2:
                psar[i] = max(psar[i], high[i - 2])
            if high[i] > psar[i]:
                bull    = True
                psar[i] = ep
                ep      = high[i]
                af      = af_start
            else:
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + af_inc, af_max)

        psar_bullish[i] = bull

    df = df.copy()
    df["psar"]         = psar
    df["psar_bullish"] = psar_bullish
    return df


def calc_volume_heatmap(df: pd.DataFrame,
                        length: int = 60,
                        slength: int = 60,
                        t_extra_high: float = 4.0,
                        t_high:       float = 2.5,
                        t_medium:     float = 1.0,
                        t_normal:     float = -0.5) -> pd.Series:
    """Volume heatmap classifier matching buy bot logic."""
    mean      = df["volume"].rolling(window=length).mean()
    std       = df["volume"].rolling(window=slength).std()
    stdbar    = (df["volume"] - mean) / std
    direction = df["close"] > df["open"]

    def classify(row):
        if pd.isna(row["stdbar"]):
            return "Insufficient Data"
        tag = " Up" if row["direction"] else " Down"
        if row["stdbar"] > t_extra_high:
            return "Extra High Volume" + tag
        elif row["stdbar"] > t_high:
            return "High Volume" + tag
        elif row["stdbar"] > t_medium:
            return "Medium Volume" + tag
        elif row["stdbar"] > t_normal:
            return "Normal Volume" + tag
        else:
            return "Low Volume" + tag

    tmp = pd.DataFrame({"stdbar": stdbar, "direction": direction})
    return tmp.apply(classify, axis=1)


def calc_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Run all indicators on a bars DataFrame."""
    df = df.copy()
    df["ema_9"]   = calc_ema(df["close"], 9)
    df["ema_21"]  = calc_ema(df["close"], 21)
    df["ema_200"] = calc_ema(df["close"], 200)
    df = calc_psar(df)
    df["vol_heatmap"]     = calc_volume_heatmap(df)
    df["ema_9_21_bull"]   = (df["close"] > df["ema_9"]) & (df["ema_9"] > df["ema_21"])
    df["ema_200_bull"]    = df["close"] > df["ema_200"]

    # Volume z-score (simple, vs last 20 bars)
    vol_mean  = df["volume"].rolling(20).mean()
    vol_std   = df["volume"].rolling(20).std().replace(0, 1)
    df["vol_z"] = (df["volume"] - vol_mean) / vol_std
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SERIALIZATION HELPER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(obj):
    """
    Recursively convert numpy scalars to native Python types so that
    pymongo's BSON encoder never sees np.bool_, np.int64, np.float64, etc.
    """
    if isinstance(obj, dict):
        return {k: sanitize_for_mongo(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_for_mongo(v) for v in obj]
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# ══════════════════════════════════════════════════════════════════════════════
# SESSION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _get_et_now() -> datetime:
    utc_now = datetime.now(timezone.utc)
    offset  = timedelta(hours=-4) if 3 <= utc_now.month <= 10 else timedelta(hours=-5)
    return utc_now + offset


def is_aftermarket() -> bool:
    et = _get_et_now()
    rth_open  = et.replace(hour=9,  minute=30, second=0, microsecond=0)
    rth_close = et.replace(hour=16, minute=0,  second=0, microsecond=0)
    return not (rth_open <= et < rth_close)


def is_scan_window() -> bool:
    """Only scan between 4 AM and 8 PM ET."""
    et = _get_et_now()
    return 4 <= et.hour < 20


# ══════════════════════════════════════════════════════════════════════════════
# SMART SCANNER
# ══════════════════════════════════════════════════════════════════════════════

class SmartScanner:
    """
    Proactive market scanner — finds breakout candidates autonomously.

    Lessons from trade document built in:
      - KPRX: skip/delay if first bar >10%
      - CUPR: avoid if first bar >50%
      - STIM: bar-over-bar close confirmation
      - MGRT: allow tech-only entry (no volume requirement)
      - RDGT: trend_start_confidence instead of time-based entry
      - FCUV: momentum type for large-first-bar continuations
    """

    def __init__(self, ib: IB, db):
        self.ib          = ib
        self.watchlist   = db[WATCHLIST_COLLECTION]
        self.scan_log    = db[SCAN_LOG_COLLECTION]
        self.cooldowns   = db[COOLDOWN_COLLECTION]
        self._seen_today: set[str] = set()

    # ── IBKR Scanner ──────────────────────────────────────────────────────────

    def fetch_ibkr_candidates(self) -> list[str]:
        """Run IBKR's built-in scanners and return deduplicated symbol list."""
        symbols = set()

        for scan_code in SCAN_CODES:
            try:
                sub = ScannerSubscription(
                    instrument="STK",
                    locationCode="STK.US.MAJOR",
                    scanCode=scan_code,
                )
                tag_values = [
                    TagValue("priceAbove",  str(PRICE_MIN)),
                    TagValue("priceBelow",  str(PRICE_MAX)),
                    TagValue("volumeAbove", str(VOLUME_MIN)),
                ]
                results = self.ib.reqScannerData(sub, [], tag_values)

                batch = []
                for r in results[:SCAN_MAX_RESULTS]:
                    sym = r.contractDetails.contract.symbol
                    batch.append(sym)

                log.info(f"IBKR scan [{scan_code}]: {len(batch)} results "
                         f"→ {', '.join(batch[:8])}{'...' if len(batch) > 8 else ''}")
                symbols.update(batch)

            except Exception as e:
                log.warning(f"IBKR scan [{scan_code}] failed: {e}")

        # Filter out symbols in cooldown
        qualified = []
        for sym in symbols:
            if not self._is_in_cooldown(sym):
                qualified.append(sym)
            else:
                log.debug(f"{sym}: skipping — in sell-bot cooldown")

        return qualified

    def _is_in_cooldown(self, symbol: str) -> bool:
        record = self.cooldowns.find_one({"symbol": symbol})
        if not record:
            return False
        expires_at = record.get("expires_at")
        if not expires_at:
            return False
        now = datetime.now(timezone.utc)
        if expires_at.tzinfo is None:
            return datetime.now() < expires_at
        return now < expires_at

    # ── Per-Symbol Deep Scan ──────────────────────────────────────────────────

    def _get_bars(self, symbol: str) -> Optional[pd.DataFrame]:
        """Fetch 5-min bars from IBKR and return as DataFrame."""
        contract = Stock(symbol, "SMART", "USD")
        try:
            self.ib.qualifyContracts(contract)
        except Exception as e:
            log.warning(f"{symbol}: qualifyContracts failed — {e}")
            return None

        for what in ("TRADES", "MIDPOINT"):
            try:
                raw = self.ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr=BARS_LOOKBACK,
                    barSizeSetting=BAR_SIZE,
                    whatToShow=what,
                    useRTH=False,
                    keepUpToDate=False,
                )
                if raw:
                    df = util.df(raw)
                    df.columns = [c.lower() for c in df.columns]
                    return df
            except Exception as e:
                log.debug(f"{symbol}: reqHistoricalData [{what}] error — {e}")

        return None

    def _calc_first_bar_move(self, df: pd.DataFrame) -> tuple[float, int]:
        """
        Returns (first_bar_move_pct, bars_since_session_open).

        Finds the first bar where volume > 2× rolling avg — this is the
        "breakout bar". Returns its % move and how many bars ago it was.
        """
        if len(df) < 2:
            return 0.0, 0

        avg_vol   = df["volume"].rolling(20).mean()
        threshold = avg_vol * 2
        big_mask  = df["volume"] > threshold

        if big_mask.any():
            first_idx   = big_mask.idxmax()
            first_bar   = df.loc[first_idx]
            session_open = df.iloc[0]["open"]
            move_pct    = abs(first_bar["close"] - session_open) / (session_open + 1e-9)
            bars_ago    = len(df) - 1 - df.index.get_loc(first_idx)
            return move_pct, bars_ago

        return 0.0, 0

    def _calc_trend_start_confidence(self, df: pd.DataFrame) -> float:
        """
        RDGT fix: multi-factor trend start detection — not time-based.
        Score 0-100.
        """
        if len(df) < 3:
            return 0.0

        score = 0.0
        last  = df.iloc[-1]
        prev  = df.iloc[-2]

        # Factor 1: Fresh PSAR bullish flip (30 pts)
        if last.get("psar_bullish", False) and not prev.get("psar_bullish", True):
            score += 30
        elif last.get("psar_bullish", False):
            score += 15  # Already bullish

        # Factor 2: EMA alignment (30 pts)
        if last["close"] > last["ema_9"]:    score += 10
        if last["ema_9"]  > last["ema_21"]:  score += 10
        if last["ema_21"] > last["ema_200"]: score += 10

        # Factor 3: Volume z-score (20 pts)
        vol_z = last.get("vol_z", 0)
        score += min(20, max(0, vol_z * 5))

        # Factor 4: 5-bar momentum (20 pts)
        if len(df) >= 6:
            roc = (df["close"].iloc[-1] - df["close"].iloc[-6]) / (df["close"].iloc[-6] + 1e-9) * 100
            score += min(20, max(0, roc * 2))

        return min(100.0, score)

    def _detect_consolidation(self, df: pd.DataFrame) -> bool:
        """
        KPRX fix: tight range + declining volume after a big move =
        consolidation phase. Signals a safe pullback re-entry opportunity.
        """
        if len(df) < CONSOLIDATION_BARS + 1:
            return False

        recent = df.tail(CONSOLIDATION_BARS)
        close_avg    = recent["close"].mean()
        range_pct    = (recent["high"].max() - recent["low"].min()) / (close_avg + 1e-9)
        vol_declining = recent["volume"].iloc[-1] < recent["volume"].iloc[0]
        return range_pct < CONSOLIDATION_RANGE_PCT and vol_declining

    def _bar_over_bar_close(self, df: pd.DataFrame) -> bool:
        """STIM fix: last bar close > previous bar close."""
        if len(df) < 2:
            return False
        return float(df["close"].iloc[-1]) > float(df["close"].iloc[-2])

    def _compute_entry_score(self, row: pd.Series) -> tuple[int, dict]:
        """
        Same 3-point scoring as buy bot:
          1 pt: PSAR bullish
          1 pt: Extra High Volume Up
          1 pt: Close > EMA9 > EMA21
        """
        cond_psar   = bool(row.get("psar_bullish", False))
        cond_volume = str(row.get("vol_heatmap", "")).startswith("Extra High Volume Up")
        cond_ema    = bool(row.get("ema_9_21_bull", False))

        details = {
            "psar_bullish":      cond_psar,
            "extra_high_vol_up": cond_volume,
            "ema_9_21_stack":    cond_ema,
        }
        return int(cond_psar) + int(cond_volume) + int(cond_ema), details

    def scan_single(self, symbol: str) -> Optional[ScanResult]:
        """Deep analysis of a single symbol. Returns ScanResult or None."""
        df = self._get_bars(symbol)
        if df is None or len(df) < MIN_BARS_FOR_SCAN:
            log.debug(f"{symbol}: insufficient bars — skipping")
            return None

        df = calc_all_indicators(df)
        latest = df.iloc[-1]
        prev   = df.iloc[-2] if len(df) > 1 else latest

        current_price           = float(latest["close"])
        first_bar_pct, bars_ago = self._calc_first_bar_move(df)
        trend_confidence        = self._calc_trend_start_confidence(df)
        consolidation           = self._detect_consolidation(df)
        bar_over_bar            = self._bar_over_bar_close(df)
        aftermkt                = is_aftermarket()
        entry_score, entry_det  = self._compute_entry_score(latest)
        vol_z                   = float(latest.get("vol_z", 0))
        vol_label               = str(latest.get("vol_heatmap", ""))

        # ── Entry Recommendation Logic ────────────────────────────────────────
        # CUPR lesson: >50% first bar = avoid
        if first_bar_pct > FIRST_BAR_AVOID_PCT:
            rec       = "avoid"
            e_type    = "avoid"
            risk      = "extreme"

        # Aftermarket early bars: CUPR/RDGT lesson — wait
        elif aftermkt and bars_ago < AFTERMARKET_WAIT_BARS:
            rec    = "wait_pullback"
            e_type = "pullback"
            risk   = "high"

        # KPRX lesson: first bar 10-50% — check if momentum or need pullback
        elif first_bar_pct > FIRST_BAR_WAIT_PCT:
            if bar_over_bar and bool(latest.get("psar_bullish", False)):
                # FCUV: momentum continuing even after big first bar
                rec    = "cautious"
                e_type = "momentum"
                risk   = "medium"
            elif consolidation:
                # KPRX: waited, now consolidating — ready for pullback entry
                rec    = "wait_pullback"
                e_type = "pullback"
                risk   = "medium"
            else:
                rec    = "wait_pullback"
                e_type = "pullback"
                risk   = "high"

        # Standard conditions: SMX/AIXI/MGRT
        else:
            if entry_score >= 2 and trend_confidence >= TREND_CONFIDENCE_MIN:
                rec    = "immediate"
                e_type = "standard"
                risk   = "low" if vol_z >= VOLUME_Z_STRONG else "medium"
            elif entry_score >= 1 and bool(latest.get("psar_bullish", False)) and bool(latest.get("ema_9_21_bull", False)):
                # MGRT: volume optional if technicals strong
                rec    = "immediate"
                e_type = "standard"
                risk   = "medium"
            else:
                rec    = "avoid"
                e_type = "avoid"
                risk   = "high"

        result = ScanResult(
            symbol                 = symbol,
            scan_time              = datetime.now(timezone.utc),
            current_price          = current_price,
            first_bar_move_pct     = first_bar_pct,
            volume_z_score         = vol_z,
            psar_aligned           = bool(latest.get("psar_bullish", False)),
            ema_9_21_aligned       = bool(latest.get("ema_9_21_bull", False)),
            ema_200_aligned        = bool(latest.get("ema_200_bull", False)),
            vol_heatmap_label      = vol_label,
            entry_recommendation   = rec,
            entry_type             = e_type,
            risk_level             = risk,
            trend_start_confidence = trend_confidence,
            consolidation_detected = consolidation,
            bar_over_bar_close     = bar_over_bar,
            aftermarket            = aftermkt,
            bars_since_session_open = bars_ago,
            entry_score            = entry_score,
            entry_details          = entry_det,
        )

        log.info(
            f"{symbol}: ${current_price:.2f} | rec={rec} | type={e_type} | "
            f"risk={risk} | score={entry_score}/3 | conf={trend_confidence:.0f} | "
            f"PSAR={'✓' if result.psar_aligned else '✗'} | "
            f"Vol={vol_label[:20]} | "
            f"EMA={'✓' if result.ema_9_21_aligned else '✗'} | "
            f"firstBar={first_bar_pct*100:.1f}% | "
            f"consol={'✓' if consolidation else '✗'}"
        )

        return result

    # ── Full Scan Run ─────────────────────────────────────────────────────────

    def run_scan(self) -> list[ScanResult]:
        """Execute full scan: IBKR scanner → per-symbol analysis → sorted results."""
        log.info("=" * 60)
        log.info(f"SCANNER RUN — {datetime.now().strftime('%H:%M:%S')}")

        candidates = self.fetch_ibkr_candidates()
        log.info(f"Candidates from IBKR scanner: {len(candidates)}")

        results = []
        for symbol in candidates:
            try:
                result = self.scan_single(symbol)
                if result and result.entry_recommendation != "avoid":
                    results.append(result)
            except Exception as e:
                log.warning(f"{symbol}: scan error — {e}")

        # RDGT fix: sort by trend_start_confidence, not time
        results.sort(key=lambda x: x.trend_start_confidence, reverse=True)

        log.info(f"Scan complete: {len(results)} actionable candidates "
                 f"(from {len(candidates)} total)")
        return results

    # ── MongoDB Watchlist Writer ───────────────────────────────────────────────

    def _priority_for_rec(self, rec: str) -> int:
        return {"immediate": 10, "cautious": 8, "wait_pullback": 5}.get(rec, 0)

    def write_to_watchlist(self, results: list[ScanResult]):
        """Upsert scan results into MongoDB watchlist_tickers."""
        written = 0
        for r in results:
            priority = self._priority_for_rec(r.entry_recommendation)
            doc = {
                "symbol":                 r.symbol,
                "source":                 "scanner",
                "priority":               priority,
                "added_at":               r.scan_time,
                "active":                 True,
                "entry_type":             r.entry_type,
                "entry_score":            r.entry_score,
                "entry_details":          r.entry_details,
                "entry_recommendation":   r.entry_recommendation,
                "risk_level":             r.risk_level,
                "trend_start_confidence": r.trend_start_confidence,
                "first_bar_move_pct":     r.first_bar_move_pct,
                "consolidation_detected": r.consolidation_detected,
                "bar_over_bar_close":     r.bar_over_bar_close,
                "current_price":          r.current_price,
                "volume_z_score":         r.volume_z_score,
                "vol_heatmap_label":      r.vol_heatmap_label,
                "psar_aligned":           r.psar_aligned,
                "ema_9_21_aligned":       r.ema_9_21_aligned,
                "ema_200_aligned":        r.ema_200_aligned,
                # Buy bot: only enter immediately if score ≥ this
                "entry_score_requirement": 2 if priority >= 8 else 3,
            }
            self.watchlist.update_one(
                {"symbol": r.symbol},
                {"$set": sanitize_for_mongo(doc)},
                upsert=True,
            )
            written += 1

        log.info(f"Watchlist updated: {written} symbols written/refreshed")

    def log_scan_run(self, results: list[ScanResult]):
        """Persist scan run summary to MongoDB for audit trail."""
        self.scan_log.insert_one({
            "run_time":      datetime.now(timezone.utc),
            "total_found":   len(results),
            "immediate":     sum(1 for r in results if r.entry_recommendation == "immediate"),
            "cautious":      sum(1 for r in results if r.entry_recommendation == "cautious"),
            "wait_pullback": sum(1 for r in results if r.entry_recommendation == "wait_pullback"),
            "symbols":       [r.symbol for r in results],
        })


# ══════════════════════════════════════════════════════════════════════════════
# EOD CLEANUP
# ══════════════════════════════════════════════════════════════════════════════

def clear_watchlist_eod(watchlist_col):
    et = _get_et_now()
    if et.hour >= 16:
        result = watchlist_col.delete_many({"source": "scanner"})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} scanner-sourced ticker(s)")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=" * 60)
    log.info("SCANNER BOT v1.0 Starting")
    log.info(f"Scan codes: {SCAN_CODES}")
    log.info(f"Price range: ${PRICE_MIN}–${PRICE_MAX} | Min vol: {VOLUME_MIN:,}")
    log.info(f"Scan interval: {SCAN_INTERVAL_SEC}s")
    log.info("=" * 60)

    # IBKR
    util.startLoop()
    ib = IB()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")

    # MongoDB
    mongo = MongoClient(MONGO_URI)
    db    = mongo[DB_NAME]

    scanner = SmartScanner(ib, db)

    try:
        while True:
            if not is_scan_window():
                log.info("Outside scan window (4 AM–8 PM ET) — sleeping 5 min")
                time.sleep(300)
                continue

            clear_watchlist_eod(db[WATCHLIST_COLLECTION])

            results = scanner.run_scan()
            scanner.write_to_watchlist(results)
            scanner.log_scan_run(results)

            if results:
                log.info("Top 5 candidates:")
                for r in results[:5]:
                    log.info(
                        f"  {r.symbol:6s} ${r.current_price:.2f} | "
                        f"rec={r.entry_recommendation:14s} | "
                        f"conf={r.trend_start_confidence:.0f} | "
                        f"score={r.entry_score}/3 | risk={r.risk_level}"
                    )

            log.info(f"Next scan in {SCAN_INTERVAL_SEC}s...")
            time.sleep(SCAN_INTERVAL_SEC)

    except KeyboardInterrupt:
        log.info("Scanner bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


2026-04-08 06:53:54,318 [INFO] ============================================================
2026-04-08 06:53:54,319 [INFO] SCANNER BOT v1.0 Starting
2026-04-08 06:53:54,320 [INFO] Scan codes: ['TOP_PERC_GAIN', 'HIGH_RELATIVE_VOLUME', 'MOST_ACTIVE']
2026-04-08 06:53:54,321 [INFO] Price range: $0.5–$20.0 | Min vol: 300,000
2026-04-08 06:53:54,323 [INFO] Scan interval: 120s
2026-04-08 06:53:54,325 [INFO] ============================================================
2026-04-08 06:53:54,477 [INFO] IBKR connected | accounts: ['DUD087866']
2026-04-08 06:53:54,484 [INFO] ============================================================
2026-04-08 06:53:54,485 [INFO] SCANNER RUN — 06:53:54
2026-04-08 06:53:54,837 [INFO] IBKR scan [TOP_PERC_GAIN]: 30 results → UCAR, VSME, IPW, OKLL, IRIX, BMNU, IRE, CRCG...


Error 162, reqId 3: Historical Market Data Service error message:API scanner subscription cancelled: 3
Error 162, reqId 4: Historical Market Data Service error message:Scanner type with code HIGH_RELATIVE_VOLUME is disabled.


2026-04-08 06:53:54,987 [INFO] IBKR scan [HIGH_RELATIVE_VOLUME]: 0 results → 


Error 365, reqId 4: No scanner subscription found for ticker id:4


2026-04-08 06:53:55,385 [INFO] IBKR scan [MOST_ACTIVE]: 30 results → AIXI, IPW, UCAR, VSME, TSLL, UVIX, SCO, NOK...
2026-04-08 06:53:55,414 [INFO] Candidates from IBKR scanner: 46


...

2026-04-08 16:21:32,176 [INFO] IBKR scan [MOST_ACTIVE]: 30 results → UCAR, OMEX, AIXI, TSLL, BMNU, TZA, NOK, AAL...
2026-04-08 16:21:32,196 [INFO] Candidates from IBKR scanner: 30


Error 162, reqId 26733: Historical Market Data Service error message:API scanner subscription cancelled: 26733


2026-04-08 16:21:32,450 [INFO] PLUG: $2.66 | rec=immediate | type=standard | risk=medium | score=2/3 | conf=36 | PSAR=✓ | Vol=Low Volume Down | EMA=✓ | firstBar=0.7% | consol=✓
2026-04-08 16:21:32,748 [INFO] BBGI: $5.55 | rec=avoid | type=avoid | risk=extreme | score=0/3 | conf=0 | PSAR=✗ | Vol=Low Volume Down | EMA=✗ | firstBar=59.7% | consol=✗
2026-04-08 16:21:33,011 [INFO] AIXI: $1.30 | rec=avoid | type=avoid | risk=high | score=0/3 | conf=0 | PSAR=✗ | Vol=Low Volume Down | EMA=✗ | firstBar=3.4% | consol=✗
2026-04-08 16:21:33,256 [INFO] F: $12.18 | rec=avoid | type=avoid | risk=high | score=1/3 | conf=35 | PSAR=✓ | Vol=Low Volume Down | EMA=✗ | firstBar=0.6% | consol=✓
2026-04-08 16:21:33,416 [INFO] SVC: $1.32 | rec=immediate | type=standard | risk=medium | score=2/3 | conf=50 | PSAR=✓ | Vol=Normal Volume Down | EMA=✓ | firstBar=0.8% | consol=✗
2026-04-08 16:21:33,628 [INFO] UVIX: $6.85 | rec=avoid | type=avoid | risk=high | score=0/3 | conf=12 | PSAR=✗ | Vol=Low Volume Down | EMA=✗

Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Peer closed connection.


2026-04-08 19:17:33,853 [WARNING] IBKR scan [TOP_PERC_GAIN] failed: Socket disconnect
2026-04-08 19:17:33,871 [WARNING] IBKR scan [HIGH_RELATIVE_VOLUME] failed: Not connected
2026-04-08 19:17:33,875 [WARNING] IBKR scan [MOST_ACTIVE] failed: Not connected
2026-04-08 19:17:33,879 [INFO] Candidates from IBKR scanner: 0
2026-04-08 19:17:33,881 [INFO] Scan complete: 0 actionable candidates (from 0 total)
2026-04-08 19:17:33,886 [INFO] Watchlist updated: 0 symbols written/refreshed
2026-04-08 19:17:33,911 [INFO] Next scan in 120s...
